# 3. Compare Churn Models

`Churn` is 1 for customers who left and 0 for customers who stayed. This notebook prepares the data, compares three classifiers, and reports the best model.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

## Prepare the data

Turn text answers into indicator columns, split into training and test data, then fill missing values using medians from the training data.

In [2]:
PROJECT_DIR = Path.cwd().parent
data_file = PROJECT_DIR / "Cleaned_Datasets" / "telco_customer_churn.csv"
data = pd.read_csv(data_file)

X = pd.get_dummies(data.drop(columns="Churn"), drop_first=True)
y = data["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

medians = X_train.median()
X_train = X_train.fillna(medians)
X_test = X_test.fillna(medians)

print("Training rows:", len(y_train))
print("Test rows:", len(y_test))

Training rows: 5634
Test rows: 1409


## Train and compare three models

F1 balances precision and recall for customers who churn. We train each model, calculate its test F1, and compare the scores.

In [3]:
models = {
    "Logistic regression": LogisticRegression(
        C=0.1, class_weight="balanced", max_iter=3000
    ),
    "Random forest": RandomForestClassifier(
        n_estimators=400, max_depth=12,
        class_weight="balanced", random_state=42
    ),
    "Gradient boosting": HistGradientBoostingClassifier(
        learning_rate=0.1, max_iter=100, max_leaf_nodes=15,
        class_weight="balanced", random_state=42
    ),
}

model_scores = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    model_scores[name] = f1_score(y_test, predictions)

score_table = pd.DataFrame({
    "Model": model_scores.keys(),
    "Test F1": model_scores.values(),
}).sort_values("Test F1", ascending=False)
display(score_table)

,Model,Test F1
2,Gradient boosting,0.636656
0,Logistic regression,0.619247
1,Random forest,0.617647


## Evaluate the best model

In [4]:
best_name = score_table.iloc[0]["Model"]
best_model = models[best_name]
best_predictions = best_model.predict(X_test)

print("Best model:", best_name)
print("Confusion matrix:")
print(confusion_matrix(y_test, best_predictions))
print("\nClassification report:")
print(classification_report(
    y_test, best_predictions, target_names=["Stayed", "Churned"], zero_division=0
))

Best model: Gradient boosting
Confusion matrix:
[[773 262]
 [ 77 297]]

Classification report:
              precision    recall  f1-score   support

      Stayed       0.91      0.75      0.82      1035
     Churned       0.53      0.79      0.64       374

    accuracy                           0.76      1409
   macro avg       0.72      0.77      0.73      1409
weighted avg       0.81      0.76      0.77      1409

